# 39. 세그먼트 트리와 펜윅 트리

경시 · 문제 191–195

**사용법**
1. 바로 아래 **🔧 채점기 준비** 셀을 먼저 실행합니다(▶).
2. 각 문제의 `%%writefile` 셀에 코드를 작성하고 실행하면 파일로 저장됩니다.
3. 그 아래 `check(번호)` 셀을 실행하면 채점됩니다.
4. 직접 입력을 넣어 실행해 보려면 새 셀에 `%run p001.py`처럼 입력합니다.

> 정답과 해설은 위키독스 본문의 접힌 상자에 있습니다. 충분히 고민한 뒤에 펼쳐 보세요.

> 제3부의 시간 제한은 Colab 기준입니다. 숨은 테스트에는 제한 범위의 큰 입력이 들어 있으므로, 효율이 낮은 풀이는 '시간 초과'가 납니다.

In [ ]:
#@title 🔧 채점기 준비 (먼저 실행하세요)
TESTS = {191: [['5 4\n1 2 3 4 5\n2 2 4\n1 3 10\n2 1 5\n2 3 3', '9\n22\n10', True, None], ['1 2\n5\n1 1 -5\n2 1 1', '-5', False, None], [None, 'sha256:35fe76e509ac9fc93cba17944d8f41f6afb7c83c988b40c783c3b1f23e69e4a7', False, 'import random\nrandom.seed(225)\nn = q = 100000\nprint(n, q)\nprint(*[random.randint(-10**9, 10**9) for _ in range(n)])\nfor _ in range(q):\n    if random.random() < 0.5:\n        print(1, random.randint(1, n), random.randint(-10**9, 10**9))\n    else:\n        l = random.randint(1, n); r = random.randint(l, n)\n        print(2, l, r)']], 192: [['6 4\n5 2 8 6 3 7\n2 1 3\n2 4 6\n1 2 9\n2 1 3', '2\n3\n5', True, None], ['1 2\n5\n1 1 -5\n2 1 1', '-5', False, None], [None, 'sha256:fc24dba2a32e45a21ed10e1283ad489cb0d65f0a150c102c84803bd98ceaa410', False, 'import random\nrandom.seed(226)\nn = q = 100000\nprint(n, q)\nprint(*[random.randint(-10**9, 10**9) for _ in range(n)])\nfor _ in range(q):\n    if random.random() < 0.5:\n        print(1, random.randint(1, n), random.randint(-10**9, 10**9))\n    else:\n        l = random.randint(1, n); r = random.randint(l, n)\n        print(2, l, r)']], 193: [['6\n1 5\n1 3\n1 5\n3 2\n2 3\n3 1', '5\n5', True, None], ['2\n1 1000000\n3 1', '1000000', False, None], [None, 'sha256:4935c6bd105728a19fa32aa6356df29aac5bb6c380271a5f4f47d69dbbdc38cc', False, 'import random\nrandom.seed(227)\nq = 100000\nprint(q)\nbox = []\nimport bisect\nfor _ in range(q):\n    r = random.random()\n    if r < 0.5 or not box:\n        x = random.randint(1, 10**6)\n        bisect.insort(box, x)\n        print(1, x)\n    elif r < 0.75:\n        x = box.pop(random.randrange(len(box)))\n        print(2, x)\n    else:\n        k = random.randint(1, len(box))\n        box.pop(k - 1)\n        print(3, k)']], 194: [['5 4\n1 2 3 4 5\n1 2 4 10\n2 1 5\n1 1 1 -1\n2 1 2', '45\n12', True, None], ['1 3\n0\n1 1 1 5\n2 1 1\n2 1 1', '5\n5', False, None], [None, 'sha256:5f058f124102b450df1a1d0e78762a54cc6d77c6795e5a2e899b503722523ab7', False, 'import random\nrandom.seed(228)\nn = q = 100000\nprint(n, q)\nprint(*[random.randint(-10**6, 10**6) for _ in range(n)])\nfor _ in range(q):\n    l = random.randint(1, n); r = random.randint(l, n)\n    if random.random() < 0.5:\n        print(1, l, r, random.randint(-10**6, 10**6))\n    else:\n        print(2, l, r)']], 195: [['6\n1 2 1 3 2 1\n4\n1 3\n2 5\n1 6\n4 4', '2\n3\n3\n1', True, None], ['1\n7\n1\n1 1', '1', False, None], [None, 'sha256:c02e3cec5113fe0074da8800b35ed72b56afc9697353e3e3aedb5e42bfb73b96', False, 'import random\nrandom.seed(229)\nn = 100000\nprint(n)\nprint(*[random.randint(1, 1000) for _ in range(n)])\nq = 100000\nprint(q)\nfor _ in range(q):\n    l = random.randint(1, n); r = random.randint(l, n)\n    print(l, r)']]}
TL = {191: 3, 192: 3, 193: 3, 194: 3, 195: 3}
import subprocess, sys, pathlib, hashlib

_GEN_CACHE = {}

def _norm(s):
    return "\n".join(l.rstrip() for l in s.strip("\n").splitlines()).strip("\n")

def _input_of(inp, gen):
    if gen is None:
        return inp
    if gen not in _GEN_CACHE:
        _GEN_CACHE[gen] = subprocess.run([sys.executable, "-c", gen], capture_output=True, text=True).stdout
    return _GEN_CACHE[gen]

def check(pid):
    """p{번호}.py 를 모든 테스트로 실행해 채점합니다."""
    path = pathlib.Path(f"p{pid:03d}.py")
    if not path.exists():
        print(f"먼저 위의 %%writefile p{pid:03d}.py 셀을 실행하세요."); return
    limit = TL[pid]
    ok = 0
    for i, (inp, exp, shown, gen) in enumerate(TESTS[pid], 1):
        tag = f"예제 {i}" if shown else f"숨은 테스트 {i}"
        data = _input_of(inp, gen)
        try:
            r = subprocess.run([sys.executable, str(path)], input=data.rstrip("\n") + "\n",
                               capture_output=True, text=True, timeout=limit)
        except subprocess.TimeoutExpired:
            print(f"⏱ {tag}: 시간 초과 ({limit}초)"); continue
        got = _norm(r.stdout)
        if exp.startswith("sha256:"):
            got_cmp = "sha256:" + hashlib.sha256(got.encode()).hexdigest()
        else:
            got_cmp = got
        if r.returncode:
            last = r.stderr.strip().splitlines()[-1] if r.stderr.strip() else "알 수 없는 오류"
            print(f"💥 {tag}: 실행 오류 → {last}")
        elif got_cmp == exp:
            ok += 1; print(f"✅ {tag}")
        elif shown:
            print(f"❌ {tag}\n   입력: {inp!r}\n   기대: {exp!r}\n   실제: {got!r}")
        elif gen is not None:
            print(f"❌ {tag}  (큰 입력)")
        else:
            print(f"❌ {tag}  (입력: {inp[:60]!r})")
    total = len(TESTS[pid])
    print("─" * 30)
    print(f"🎉 {pid:03d}번 통과! ({ok}/{total})" if ok == total else f"{ok}/{total} 통과 — 다시 도전해 보세요.")
print('채점기 준비 완료!')

## 191. 펜윅 트리

길이 N의 수열에 Q개의 명령을 처리하십시오. `1 i v`는 i번째 원소를 v로 **바꾸고**, `2 l r`은 l~r번째 원소의 합을 출력합니다.

**입력** 첫째 줄에 N과 Q, 둘째 줄에 N개의 정수, 이어서 Q줄에 명령이 주어집니다.

**출력** `2` 명령마다 한 줄에 답을 출력합니다.

**제한** 1 ≤ N, Q ≤ 10⁵, |원소|, |v| ≤ 10⁹, 시간 제한 3초

**예제 1**
입력
```
5 4
1 2 3 4 5
2 2 4
1 3 10
2 1 5
2 3 3
```
출력
```
9
22
10
```

In [ ]:
%%writefile p191.py
# 여기에 코드를 작성하세요


In [ ]:
check(191)

## 192. 구간 최솟값

길이 N의 수열에 Q개의 명령을 처리하십시오. `1 i v`는 i번째 원소를 v로 바꾸고, `2 l r`은 l~r번째 원소의 최솟값을 출력합니다.

**입력** 첫째 줄에 N과 Q, 둘째 줄에 N개의 정수, 이어서 Q줄에 명령이 주어집니다.

**출력** `2` 명령마다 한 줄에 답을 출력합니다.

**제한** 1 ≤ N, Q ≤ 10⁵, |원소|, |v| ≤ 10⁹, 시간 제한 3초

**예제 1**
입력
```
6 4
5 2 8 6 3 7
2 1 3
2 4 6
1 2 9
2 1 3
```
출력
```
2
3
5
```

In [ ]:
%%writefile p192.py
# 여기에 코드를 작성하세요


In [ ]:
check(192)

## 193. K번째 사탕

빈 상자에 Q개의 명령을 처리하십시오. `1 x`는 맛 등급 x인 사탕을 하나 넣고, `2 x`는 맛 등급 x인 사탕을 하나 꺼내며(반드시 있음), `3 k`는 등급이 낮은 쪽부터 k번째 사탕의 등급을 출력하고 그 사탕을 꺼냅니다(k번째 사탕은 반드시 있음).

**입력** 첫째 줄에 Q, 이어서 Q줄에 명령이 주어집니다.

**출력** `3` 명령마다 한 줄에 답을 출력합니다.

**제한** 1 ≤ Q ≤ 10⁵, 1 ≤ x ≤ 10⁶, 시간 제한 3초

**예제 1**
입력
```
6
1 5
1 3
1 5
3 2
2 3
3 1
```
출력
```
5
5
```

In [ ]:
%%writefile p193.py
# 여기에 코드를 작성하세요


In [ ]:
check(193)

## 194. 구간 더하기, 구간 합

길이 N의 수열에 Q개의 명령을 처리하십시오. `1 l r v`는 l~r번째 원소에 모두 v를 더하고, `2 l r`은 l~r번째 원소의 합을 출력합니다.

**입력** 첫째 줄에 N과 Q, 둘째 줄에 N개의 정수, 이어서 Q줄에 명령이 주어집니다.

**출력** `2` 명령마다 한 줄에 답을 출력합니다.

**제한** 1 ≤ N, Q ≤ 10⁵, |원소|, |v| ≤ 10⁶, 시간 제한 3초

**예제 1**
입력
```
5 4
1 2 3 4 5
1 2 4 10
2 1 5
1 1 1 -1
2 1 2
```
출력
```
45
12
```

In [ ]:
%%writefile p194.py
# 여기에 코드를 작성하세요


In [ ]:
check(194)

## 195. 구간의 서로 다른 수

길이 N의 수열과 Q개의 질의 `l r`이 주어집니다. 각 질의마다 l~r번째 원소 가운데 서로 다른 값의 개수를 출력하십시오.

**입력** 첫째 줄에 N, 둘째 줄에 N개의 정수, 셋째 줄에 Q, 이어서 Q줄에 l과 r이 주어집니다.

**출력** 질의마다 한 줄에 답을 입력 순서대로 출력합니다.

**제한** 1 ≤ N, Q ≤ 10⁵, 1 ≤ 원소 ≤ 10⁹, 시간 제한 3초

**예제 1**
입력
```
6
1 2 1 3 2 1
4
1 3
2 5
1 6
4 4
```
출력
```
2
3
3
1
```

In [ ]:
%%writefile p195.py
# 여기에 코드를 작성하세요


In [ ]:
check(195)